# 7 — Type checking

**The concept:** because wiring happens by name, a mistake in a name or a type is not a syntax
error — it is a pipeline that runs and produces a wrong answer. SmartMDAO checks the **producer →
consumer edges** using your annotations, before anything executes.

Two rules govern the whole feature:

1. A declared mismatch is an error.
2. **Missing type information is not an error.** It degrades to *unchecked*, never to a failure.

In [1]:
from smartmdao import Pipeline, TypeMismatchError

bad = Pipeline()

@bad.step(outputs=["config"])
def produce(seed: int) -> dict:
    return {"seed": seed}

@bad.step(outputs=["report"])
def consume(config: str) -> str:        # expects str, gets dict
    return config.upper()

try:
    bad.run(seed=1)
except TypeMismatchError as error:
    print(f"TypeMismatchError: {error}")

Pipeline execution failed: Type mismatch on 'config': step 'produce' declares config -> dict, but step 'consume' expects config: str.


TypeMismatchError: Type mismatch on 'config': step 'produce' declares config -> dict, but step 'consume' expects config: str.


That check ran **before** either discipline was called. Nothing was computed and nothing was
wasted.

## Unannotated is unchecked, not broken

A function with no annotations is perfectly legal. It simply cannot be checked, and that is treated
as an absence of information rather than a problem.

In [2]:
untyped = Pipeline()

@untyped.step(outputs=["config"])
def produce_untyped(seed):              # no annotations at all
    return {"seed": seed}

@untyped.step(outputs=["report"])
def consume_untyped(config: str):       # would be a mismatch IF the producer declared a type
    return str(config)

print(untyped.run(seed=1))
print()
print("no error: the producer declared nothing, so there was nothing to contradict")

{'seed': 1, 'config': {'seed': 1}, 'report': "{'seed': 1}"}

no error: the producer declared nothing, so there was nothing to contradict


## Runtime checking is opt-in

Structural checking of the *declared* edges is free, so it always runs. Checking the **actual
values** on every call costs time on every step invocation — which matters inside a convergence
loop — so it is opt-in via `runtime_type_checks=True`.

In [3]:
lying = Pipeline(runtime_type_checks=True)

@lying.step(outputs=["count"])
def claims_int(x: float) -> int:
    return "not an int at all"          # the annotation is a lie

try:
    lying.run(x=1.0)
except TypeMismatchError as error:
    print(f"caught at runtime: {error}")

relaxed = Pipeline()                     # the default
relaxed.add(claims_int, outputs=["count"])
print()
print("without runtime checks:", relaxed.run(x=1.0))

Pipeline execution failed: Step 'claims_int' produced count='not an int at all' (str), expected count: int.


caught at runtime: Step 'claims_int' produced count='not an int at all' (str), expected count: int.

without runtime checks: {'x': 1.0, 'count': 'not an int at all'}


## `int` does not satisfy `float`

`StandardTypeChecker` is **strict by design**, and this is the single most common "is this a bug?"
question. It is not: strictness is what makes the check meaningful, and loosening it is a
documented extension point rather than a hidden default.

In [4]:
from smartmdao import StandardTypeChecker

checker = StandardTypeChecker()
for produced, expected in [(int, float), (float, float), (bool, int), (dict, str), (list, list)]:
    ok = checker.check_types(produced, expected)
    print(f"produces {produced.__name__:6} -> expects {expected.__name__:6}  {'ok' if ok else 'MISMATCH'}")

produces int    -> expects float   MISMATCH
produces float  -> expects float   ok
produces bool   -> expects int     ok
produces dict   -> expects str     MISMATCH
produces list   -> expects list    ok


## Writing your own checker

`TypeChecker` is a protocol. Implement it to loosen the rules — here, accepting `int` where `float`
is declared, which is the usual request.

In [5]:
from smartmdao import TypeChecker

class NumericTolerantChecker:
    """Accepts int where float is declared; otherwise defers to the standard rules."""

    def __init__(self):
        self._standard = StandardTypeChecker()

    def check_types(self, produced, expected) -> bool:
        if expected is float and produced is int:
            return True
        return self._standard.check_types(produced, expected)

    def check_value(self, value, expected) -> bool:
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            return True
        return self._standard.check_value(value, expected)

tolerant = Pipeline(type_checker=NumericTolerantChecker())

@tolerant.step(outputs=["count"])
def produce_int(seed: int) -> int:
    return seed * 2

@tolerant.step(outputs=["scaled"])
def wants_float(count: float) -> float:
    return count * 1.5

print("satisfies the TypeChecker protocol:", isinstance(NumericTolerantChecker(), TypeChecker))
print(tolerant.run(seed=4))

satisfies the TypeChecker protocol: True
{'seed': 4, 'count': 8, 'scaled': 12.0}


## What the checker deliberately does not do

Generic containers are compared on their **origin only** — `list[int]` and `list[str]` are both just
`list`. Element types are never inspected.

That is a deliberate trade: deep structural checking on every call inside a convergence loop would
cost more than it saves. Documented here so a passing check is not read as stronger than it is.

In [6]:
print("list[int] vs list[str]:", checker.check_types(list[int], list[str]),
      " <- indistinguishable, by design")

list[int] vs list[str]: True  <- indistinguishable, by design


---

**Next:** [8 — Caching](08-caching.ipynb).